# 1. Importacion y Configuracion

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.19.0


# 2. Cargar el Dataset IMDb

In [2]:
MAX_WORDS = 10000  # Solo usaremos las 10,000 palabras más frecuentes
MAX_LEN = 200      # Cortamos/rellenamos las reseñas a 200 palabras

# Cargar datos (las palabras ya son índices numéricos)
(train_data, train_labels), (test_data, test_labels) = datasets.imdb.load_data(num_words=MAX_WORDS)

# Pad sequences: Asegurar que todas las entradas tengan la misma longitud (200)
train_data = pad_sequences(train_data, maxlen=MAX_LEN)
test_data = pad_sequences(test_data, maxlen=MAX_LEN)

print(f"Forma datos entrenamiento: {train_data.shape}")
print(f"Ejemplo de reseña (índices): {train_data[0][:10]}...")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Forma datos entrenamiento: (25000, 200)
Ejemplo de reseña (índices): [  5  25 100  43 838 112  50 670   2   9]...


# 3. Construir la CNN para Texto (1D)

In [7]:
model = models.Sequential([
    # 1. Embedding: Convierte índices enteros en vectores densos (ej. vector de 128 dim)
    layers.Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),

    # 2. Convolución 1D: Busca patrones de 5 palabras (kernel_size=5)
    layers.Conv1D(filters=64, kernel_size=5, activation='relu'),

    # 3. GlobalMaxPooling: Se queda con la característica más fuerte detectada
    layers.GlobalMaxPooling1D(),

    # 4. Capas densas clásicas para clasificar
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5), # Reduce sobreajuste
    layers.Dense(1, activation='sigmoid') # Salida binaria (0=Negativa, 1=Positiva)
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 196, 64)        │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,323,137 (5.05 MB)

 Trainable params: 1,323,137 (5.05 MB)

 Non-trainable params: 0 (0.00 B)

# 4. Entrenar el Modelo

In [8]:
history = model.fit(train_data, train_labels,
                    epochs=10,
                    batch_size=128,
                    validation_split=0.2) # Usamos 20% de train para validar

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 7s 25ms/step - accuracy: 0.5978 - loss: 0.6527 - val_accuracy: 0.8204 - val_loss: 0.4141
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8465 - loss: 0.3715 - val_accuracy: 0.8820 - val_loss: 0.2944
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9330 - loss: 0.2067 - val_accuracy: 0.8874 - val_loss: 0.2770
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9746 - loss: 0.1071 - val_accuracy: 0.8890 - val_loss: 0.3056
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9910 - loss: 0.0557 - val_accuracy: 0.8876 - val_loss: 0.3447
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9967 - loss: 0.0303 - val_accuracy: 0.8876 - val_loss: 0.3948
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9986 - loss: 0.0190 - val_accuracy: 0.8850 - val_loss: 0.4449
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9991 - loss: 0.0160 - val_accuracy: 0

# 5. Evaluar y Probar

In [9]:
# Evaluar en datos de test
loss, accuracy = model.evaluate(test_data, test_labels)
print(f"\nPrecisión en test: {accuracy*100:.2f}%")

# Función para probar con tus propias frases
word_index = datasets.imdb.get_word_index()

def predict_sentiment(text):
    # Preprocesamiento manual básico
    words = text.lower().split()
    tokens = [word_index.get(w, 0) + 3 for w in words] # +3 por índices reservados
    tokens = pad_sequences([tokens], maxlen=MAX_LEN)

    pred = model.predict(tokens)[0][0]
    sentiment = "Positiva 😊" if pred > 0.5 else "Negativa 😞"
    print(f"Frase: '{text}'\nPredicción: {sentiment} ({pred:.4f})")

# ¡Pruébalo!
predict_sentiment("This movie was fantastic and I loved the acting")
predict_sentiment("Terrible script and boring direction, waste of time")

782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8767 - loss: 0.5552

Precisión en test: 87.77%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
Frase: 'This movie was fantastic and I loved the acting'
Predicción: Positiva 😊 (0.9999)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Frase: 'Terrible script and boring direction, waste of time'
Predicción: Negativa 😞 (0.0000)
